# Авито: энкодер, раунд 2 — трудные негативы

**Как запускать:** новый ноутбук, **GPU T4 ×2**, интернет не нужен, **Save Version → Save & Run All**.

**Входы (Add Input):**
- `avito_kandidates_ranking` — `prep/` и `features/` (запросы CatBoost-train и кандидаты BM25 для val)
- вывод энкодера **раунда 1** — `encoder/` (дообученная модель `model/` и векторы `item_emb.npy`, `item_ids.parquet`, `query_emb.npy`, `query_ids.parquet`)

**Что делает:**
1. Берёт модель раунда 1 и **продолжает** обучение (lr 1e-5, одна эпоха)
2. **Трудные негативы:** для каждого обучающего запроса модель раунда 1 ищет похожие объявления корпуса в его гео-пуле
   (тот же город или ближе 50 км, категория 114). Негатив берётся случайно с мест 10–60: верхние места пропускаем,
   там часто объявления, которые тоже подходят. Майнинг использует сохранённые векторы раунда 1 — пересчитывать не нужно
3. Данные как в раунде 1: train **без запросов CatBoost-train** и без текстов val
4. Оценка на val: dense@50 и потолок BM25+dense — раунд 1 (по сохранённым векторам) против раунда 2
5. Сохраняет в `/kaggle/working/encoder/` файлы **с теми же именами**, что раунд 1, — ноутбук признаков v2 примет их без изменений

**Важно для следующего шага:** в ноутбук признаков v2 подключай **только** вывод этого ноутбука, без энкодера раунда 1,
иначе `item_emb.npy` найдётся в двух местах.

### Импорты и параметры

In [1]:
import os, glob, json, time, random, gc, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)

Q_PREFIX, P_PREFIX = 'query: ', 'passage: '
Q_MAX_LEN, P_MAX_LEN, T_MAX_LEN = 48, 192, 32
PARAMS_CHARS = 300

MAX_TRAIN_PAIRS = 250_000    # пар для обучения (с трудным негативом каждая)
BATCH_SIZE = 64              # запросов в батче; объявлений в батче вдвое больше (правильные + трудные негативы)
LR, WARMUP_SHARE = 1e-5, 0.05
SCALE = 20.0
TIME_BUDGET_MIN = 80         # страховка по времени обучения (дедлайн!)
ENC_BATCH = 512
NEG_FROM, NEG_TO = 10, 60    # трудный негатив — случайное место из [NEG_FROM, NEG_TO) выдачи модели раунда 1
MINE_BATCH = 512
GEO_RADIUS_KM, K_DENSE = 50, 100
FREEZE_WORD_EMB = True
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_GPU = torch.cuda.device_count()
print(f'устройство: {DEVICE}, GPU: {N_GPU}')

def find_dir(filename, roots=('/kaggle/input', '/kaggle/working', '.')):
    for root in roots:
        hits = sorted(glob.glob(os.path.join(root, '**', filename), recursive=True))
        if hits:
            return os.path.dirname(hits[0])
    raise FileNotFoundError(filename)

PREP_DIR, FEAT_DIR = find_dir('pairs.parquet'), find_dir('features.json')
# энкодер раунда 1 ищем только во входах, чтобы не взять собственный вывод этого ноутбука
ENC1_DIR = find_dir('item_emb.npy', roots=('/kaggle/input', '.'))
MODEL1_DIR = find_dir('config.json', roots=(ENC1_DIR,))
WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
OUT_DIR = f'{WORK}/encoder_r2' if WORK == '.' else f'{WORK}/encoder'
os.makedirs(OUT_DIR, exist_ok=True)
print('prep:', PREP_DIR, '| features:', FEAT_DIR, '| энкодер раунда 1:', ENC1_DIR, '| модель:', MODEL1_DIR, '| вывод:', OUT_DIR)

устройство: cuda, GPU: 2
prep: /kaggle/input/notebooks/olegkaran/avito-kandidates-ranking/prep | features: /kaggle/input/notebooks/olegkaran/avito-kandidates-ranking/features | энкодер раунда 1: /kaggle/input/notebooks/olegkaran/encoder-finetune/encoder | модель: /kaggle/input/notebooks/olegkaran/encoder-finetune/encoder/model | вывод: /kaggle/working/encoder


## 1. Данные и тексты (как в раунде 1)

In [2]:
import pyarrow.parquet as pq
ITEM_COLS = ['item_id', 'in_corpus', 'title_norm', 'params_norm', 'item_location_id', 'item_category_id', 'lat', 'lon']
has_desc = 'desc_norm' in pq.ParquetFile(f'{PREP_DIR}/items.parquet').schema_arrow.names
items = pd.read_parquet(f'{PREP_DIR}/items.parquet', columns=ITEM_COLS + (['desc_norm'] if has_desc else []))
if not has_desc:
    items = items.merge(pd.read_parquet(f'{PREP_DIR}/items_text', columns=['item_id', 'desc_norm']), on='item_id', how='left')

queries = pd.read_parquet(f'{PREP_DIR}/queries.parquet',
                          columns=['qid', 'split', 'query_key', 'query_norm', 'search_infm_params_text',
                                   'search_location_id', 'search_category', 'loc_lat', 'loc_lon'])
pairs = pd.read_parquet(f'{PREP_DIR}/pairs.parquet', columns=['qid', 'item_id', 'split', 'in_corpus'])
cb_train_qids = set(pd.read_parquet(f'{FEAT_DIR}/train.parquet', columns=['qid'])['qid'].unique())
val_bm25 = pd.read_parquet(f'{FEAT_DIR}/val.parquet', columns=['qid', 'item_id'])
val_qids = set(val_bm25['qid'].unique())

def query_text(row):
    f = (row.search_infm_params_text or '').strip()
    return Q_PREFIX + (row.query_norm or '') + (f' | {f}' if f else '')

queries['q_text'] = [query_text(r) for r in queries.itertuples(index=False)]
items['p_text'] = (P_PREFIX + items['title_norm'].fillna('') + '. ' +
                   items['params_norm'].fillna('').str.slice(0, PARAMS_CHARS) + '. ' + items['desc_norm'].fillna(''))
items['t_text'] = P_PREFIX + items['title_norm'].fillna('')
items = items.drop(columns=['params_norm', 'desc_norm']); gc.collect()

# корпус в порядке векторов раунда 1
enc1_items = pd.read_parquet(f'{ENC1_DIR}/item_ids.parquet')['item_id']
corpus = items.set_index('item_id').loc[enc1_items.values].reset_index()
assert corpus['in_corpus'].all(), 'в векторах раунда 1 есть объявления не из корпуса'
E1 = np.load(f'{ENC1_DIR}/item_emb.npy')                       # (189 тыс., 768) float16
cid2idx = pd.Series(np.arange(len(corpus)), index=corpus.item_id)
print(f'корпус: {len(corpus):,}, векторы раунда 1: {E1.shape}')

корпус: 189,212, векторы раунда 1: (189212, 768)


## 2. Обучающие пары

Как в раунде 1: `split == 'train'` без запросов CatBoost-train, одинаковые пары «текст запроса → объявление» схлопнуты.

In [3]:
tr = (pairs[(pairs.split == 'train') & ~pairs.qid.isin(cb_train_qids)][['qid', 'item_id']]
      .merge(queries[['qid', 'q_text', 'search_location_id', 'search_category', 'loc_lat', 'loc_lon']], on='qid')
      .merge(items[['item_id', 'p_text']], on='item_id')
      .drop_duplicates(['q_text', 'item_id']))
if len(tr) > MAX_TRAIN_PAIRS:
    tr = tr.sample(MAX_TRAIN_PAIRS, random_state=SEED)
tr = tr.reset_index(drop=True)

# для каждого текста запроса — все объявления корпуса, которые по нему выбирали (их нельзя брать в негативы)
chosen_by_text = (pairs[pairs.split == 'train'].merge(queries[['qid', 'q_text']], on='qid')
                  .loc[lambda d: d.item_id.isin(cid2idx.index)]
                  .groupby('q_text')['item_id'].apply(lambda s: set(cid2idx[s].values)).to_dict())
print(f'обучающих пар: {len(tr):,}')

обучающих пар: 250,000


## 3. Модель раунда 1 (продолжаем её обучение)

In [4]:
class Encoder(nn.Module):
    def __init__(self, path):
        super().__init__()
        self.bert = AutoModel.from_pretrained(path)

    def forward(self, input_ids, attention_mask):
        with torch.autocast('cuda', dtype=torch.float16, enabled=input_ids.is_cuda):
            out = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        m = attention_mask.unsqueeze(-1).float()
        emb = (out.float() * m).sum(1) / m.sum(1).clamp(min=1e-6)
        return F.normalize(emb, dim=-1)

tokenizer = AutoTokenizer.from_pretrained(MODEL1_DIR)
encoder = Encoder(MODEL1_DIR).to(DEVICE)
if FREEZE_WORD_EMB:
    encoder.bert.get_input_embeddings().weight.requires_grad_(False)
model = nn.DataParallel(encoder) if N_GPU > 1 else encoder

def tok(texts, max_len):
    b = tokenizer(list(texts), padding=True, truncation=True, max_length=max_len, return_tensors='pt')
    return b['input_ids'].to(DEVICE), b['attention_mask'].to(DEVICE)

@torch.no_grad()
def encode(texts, max_len, batch=ENC_BATCH, desc=''):
    model.eval()
    texts = list(texts)
    order = np.argsort([len(t) for t in texts])
    out = np.zeros((len(texts), encoder.bert.config.hidden_size), dtype=np.float16)
    t0 = time.time()
    for s in range(0, len(texts), batch):
        idx = order[s:s + batch]
        out[idx] = model(*tok([texts[i] for i in idx], max_len)).cpu().numpy().astype(np.float16)
    print(f'  {desc}: {len(texts):,} текстов за {time.time() - t0:.0f} с')
    return out

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 4. Майнинг трудных негативов

Для каждого уникального текста запроса: вектор моделью раунда 1 → косинус со всем корпусом (сохранённые векторы) →
оставляем гео-пул запроса (категория 114, тот же город или ближе 50 км) → убираем объявления, которые по этому тексту
выбирали → берём случайное место из [10, 60). Если пул слишком мал, берём самое дальнее доступное место.

In [5]:
t0 = time.time()
uq = tr.drop_duplicates('q_text')[['q_text', 'search_location_id', 'loc_lat', 'loc_lon']].reset_index(drop=True)
Q1 = encode(uq.q_text, Q_MAX_LEN, desc='уникальные тексты обучающих запросов')

E_t = torch.from_numpy(E1).to(DEVICE).float()                  # один раз в float32, а не в каждом батче
C_LAT = torch.tensor(np.radians(corpus.lat.values.astype('float64')), dtype=torch.float32, device=DEVICE)
C_LON = torch.tensor(np.radians(corpus.lon.values.astype('float64')), dtype=torch.float32, device=DEVICE)
C_LOC = torch.tensor(corpus.item_location_id.values.astype('int64'), device=DEVICE)
C_OK = torch.tensor(corpus.item_category_id.values == 114, device=DEVICE)
rng = np.random.default_rng(SEED)
neg_idx = np.full(len(uq), -1, dtype=np.int64)

for s in range(0, len(uq), MINE_BATCH):
    b = uq.iloc[s:s + MINE_BATCH]
    S = torch.from_numpy(Q1[s:s + MINE_BATCH]).to(DEVICE).float() @ E_t.T             # (батч × корпус)
    lat = torch.tensor(np.radians(b.loc_lat.fillna(0).values), dtype=torch.float32, device=DEVICE)[:, None]
    lon = torch.tensor(np.radians(b.loc_lon.fillna(0).values), dtype=torch.float32, device=DEVICE)[:, None]
    a = torch.sin((C_LAT - lat) / 2) ** 2 + torch.cos(lat) * torch.cos(C_LAT) * torch.sin((C_LON - lon) / 2) ** 2
    d = 2 * 6371 * torch.asin(torch.sqrt(a.clamp(0, 1)))
    loc = torch.tensor(b.search_location_id.values.astype('int64'), device=DEVICE)[:, None]
    geo = (C_LOC == loc) | (d <= GEO_RADIUS_KM)
    geo[torch.tensor(b.loc_lat.isna().values, device=DEVICE)] = True                  # нет координат — без гео
    S = S.masked_fill(~(geo & C_OK), float('-inf'))
    top = torch.topk(S, NEG_TO, dim=1)
    top_i, top_v = top.indices.cpu().numpy(), top.values.cpu().numpy()
    for j, qt in enumerate(b.q_text.values):
        chosen = chosen_by_text.get(qt, set())
        cand = [c for c, v in zip(top_i[j], top_v[j]) if np.isfinite(v) and c not in chosen]
        if not cand:
            continue
        lo = min(NEG_FROM, len(cand) - 1)
        neg_idx[s + j] = cand[rng.integers(lo, len(cand))]
del E_t, S, d, a, geo
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

text2neg = dict(zip(uq.q_text, neg_idx))
tr['neg'] = tr.q_text.map(text2neg)
no_neg = (tr.neg < 0).mean()
tr.loc[tr.neg < 0, 'neg'] = rng.integers(0, len(corpus), (tr.neg < 0).sum())      # редкий случай: случайный негатив
tr['n_text'] = corpus.p_text.values[tr.neg.values]
tr['neg_item'] = corpus.item_id.values[tr.neg.values]
print(f'майнинг: {time.time() - t0:.0f} с; без трудного негатива (взят случайный): {no_neg:.2%}')
ex = tr.sample(5, random_state=1)
for r in ex.itertuples():
    print(f'\n{r.q_text[:80]}\n   + {r.p_text[9:90]}\n   − {r.n_text[9:90]}')

  уникальные тексты обучающих запросов: 71,809 текстов за 30 с
майнинг: 49 с; без трудного негатива (взят случайный): 0.00%

query: ремонт стоек амортизаторов | Вид услуги Автосервис, аренда
   + Восстановление авто амортизаторов и мото. Вид услуги Автосервис, аренда Тип услуг
   − Ремонт грузовых автомобилей,маз,камаз,газель. Вид услуги Автосервис, аренда Тип у

query: научная статья | Вид услуги Обучение, курсы
   + Помощь студентам в оформлении: курсовые и дипломные работы, научные статьи ринц в
   − Обучение по косметологии. Вид услуги Обучение, курсы Место оказания услуг Республ

query: сборка мебели | Тип услуги Сборка и ремонт мебели Вид услуги Ремонт и отд
   + Сборка мебели / сборщик мебели. Вид услуги Ремонт и отделка Тип услуги Сборка и р
   − Дизайн проект интерьера. Вид услуги Ремонт и отделка Тип услуги Дизайн интерьеров

query: маникюр | Где вы оказываете услуги В салоне Где вы оказываете услуги Услу
   + Маникюр, Педикюр. Вид услуги Красота, здоровье Место оказания услу

### Батчи без дубликатов

Внутри батча не повторяются ни тексты запросов, ни объявления (правильные и негативы), иначе модель наказывалась бы за правильный ответ.

In [6]:
def make_batches(q_texts, pos_items, neg_items, batch_size, seed):
    rng_ = np.random.default_rng(seed)
    pending = list(rng_.permutation(len(q_texts)))
    batches = []
    for _ in range(5):
        cur, seen_q, seen_i, leftover = [], set(), set(), []
        for i in pending:
            if q_texts[i] in seen_q or pos_items[i] in seen_i or neg_items[i] in seen_i:
                leftover.append(i); continue
            cur.append(i); seen_q.add(q_texts[i]); seen_i.add(pos_items[i]); seen_i.add(neg_items[i])
            if len(cur) == batch_size:
                batches.append(cur); cur, seen_q, seen_i = [], set(), set()
        pending = leftover + cur
        if len(pending) < batch_size:
            break
    rng_.shuffle(batches)
    return batches

batches = make_batches(tr.q_text.values, tr.item_id.values, tr.neg_item.values, BATCH_SIZE, SEED)
print(f'батчей: {len(batches):,} по {BATCH_SIZE}')

батчей: 3,890 по 64


## 5. Оценка на val (раунд 1 — по сохранённым векторам)

Та же оценка, что в ноутбуке раунда 1: гео-пул как в признаках v2, dense@50/@100 и потолок кандидатов BM25+dense.

In [7]:
C_LAT_NP = np.radians(corpus.lat.values.astype('float64')); C_LON_NP = np.radians(corpus.lon.values.astype('float64'))
C_LOC_NP = corpus.item_location_id.values; C_CAT_NP = corpus.item_category_id.values
vq = queries[queries.qid.isin(val_qids)].reset_index(drop=True)
val_pos = (pairs[(pairs.split == 'val') & pairs.in_corpus & pairs.item_id.isin(cid2idx.index)]
           .groupby('qid')['item_id'].apply(lambda s: set(cid2idx[s].values)).to_dict())
bm25_cands = val_bm25[val_bm25.item_id.isin(cid2idx.index)].groupby('qid')['item_id'].apply(
    lambda s: set(cid2idx[s].values)).to_dict()

def pool_mask(r):
    cat = (C_CAT_NP == 114) if int(r.search_category) == 114 else np.ones(len(corpus), bool)
    if pd.isna(r.loc_lat):
        return cat
    lat, lon = np.radians(r.loc_lat), np.radians(r.loc_lon)
    a = np.sin((C_LAT_NP - lat) / 2) ** 2 + np.cos(lat) * np.cos(C_LAT_NP) * np.sin((C_LON_NP - lon) / 2) ** 2
    d = 2 * 6371 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    m = cat & ((C_LOC_NP == r.search_location_id) | (d <= GEO_RADIUS_KM))
    return m if m.any() else cat

VAL_POOLS = [np.flatnonzero(pool_mask(r)) for r in vq.itertuples(index=False)]

def evaluate(item_emb, q_emb, label):
    E = torch.from_numpy(item_emb).to(DEVICE)
    res = {'dense@50': [], 'dense@100': [], 'потолок BM25': [], 'потолок BM25+dense': []}
    for i, r in enumerate(vq.itertuples(index=False)):
        pos = val_pos.get(r.qid)
        if not pos:
            continue
        pool = VAL_POOLS[i]
        s = (E[pool] @ torch.from_numpy(q_emb[i]).to(DEVICE)).float().cpu().numpy()
        top = pool[np.argsort(-s)[:K_DENSE]]
        bm = bm25_cands.get(r.qid, set())
        res['dense@50'].append(len(pos & set(top[:50])) / len(pos))
        res['dense@100'].append(len(pos & set(top)) / len(pos))
        res['потолок BM25'].append(len(pos & bm) / len(pos))
        res['потолок BM25+dense'].append(len(pos & (bm | set(top))) / len(pos))
    del E
    out = {k: float(np.mean(v)) for k, v in res.items()}
    print(f'[{label}] ' + ' | '.join(f'{k}: {v:.4f}' for k, v in out.items()))
    return out

enc1_q = pd.read_parquet(f'{ENC1_DIR}/query_ids.parquet')['qid']
Q1_ALL = np.load(f'{ENC1_DIR}/query_emb.npy')
q1_pos = pd.Series(np.arange(len(enc1_q)), index=enc1_q.values)
EVAL = {'раунд 1': evaluate(E1, Q1_ALL[q1_pos[vq.qid].values], 'раунд 1')}

[раунд 1] dense@50: 0.8359 | dense@100: 0.8846 | потолок BM25: 0.9322 | потолок BM25+dense: 0.9673


## 6. Обучение

Для батча из N запросов: объявления = N правильных + N трудных негативов. Матрица косинусов «запросы × 2N объявлений»,
правильный ответ — на диагонали первой половины. Плюс обратное направление (объявление → запрос) по правильным.

In [8]:
trainable = [p for p in encoder.parameters() if p.requires_grad]
opt = torch.optim.AdamW(trainable, lr=LR, weight_decay=0.01)
total_steps = len(batches)
warmup = max(1, int(total_steps * WARMUP_SHARE))
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: min((s + 1) / warmup, max(0.0, (total_steps - s) / max(1, total_steps - warmup))))
try:
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))

q_texts, p_texts, n_texts = tr.q_text.values, tr.p_text.values, tr.n_text.values
t0, step, run_loss, run_n = time.time(), 0, 0.0, 0
model.train()
for b in batches:
    q = model(*tok(q_texts[b], Q_MAX_LEN))
    p = model(*tok(list(p_texts[b]) + list(n_texts[b]), P_MAX_LEN))       # правильные + негативы одним проходом
    n = q.shape[0]
    labels = torch.arange(n, device=DEVICE)
    scores = SCALE * q @ p.T                                                 # (n × 2n)
    loss = (F.cross_entropy(scores, labels) + F.cross_entropy(scores[:, :n].T, labels)) / 2

    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(trainable, 1.0)
    scaler.step(opt); scaler.update(); sched.step()

    step += 1; run_loss += loss.item(); run_n += 1
    if step % 100 == 0 or step == total_steps:
        el = time.time() - t0
        print(f'шаг {step:,}/{total_steps:,} | loss {run_loss / run_n:.4f} | {el / 60:.1f} мин, '
              f'осталось ~{el / step * (total_steps - step) / 60:.0f} мин', flush=True)
        run_loss, run_n = 0.0, 0
    if time.time() - t0 > TIME_BUDGET_MIN * 60:
        print(f'лимит времени {TIME_BUDGET_MIN} мин: остановка на шаге {step}')
        break
print(f'обучение: {step:,} шагов за {(time.time() - t0) / 60:.1f} мин')
del opt, scaler; gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

шаг 100/3,890 | loss 0.4809 | 1.9 мин, осталось ~73 мин
шаг 200/3,890 | loss 0.4662 | 3.9 мин, осталось ~72 мин
шаг 300/3,890 | loss 0.4720 | 5.9 мин, осталось ~70 мин
шаг 400/3,890 | loss 0.4864 | 7.9 мин, осталось ~69 мин
шаг 500/3,890 | loss 0.4715 | 9.8 мин, осталось ~67 мин
шаг 600/3,890 | loss 0.4642 | 11.8 мин, осталось ~65 мин
шаг 700/3,890 | loss 0.4498 | 13.8 мин, осталось ~63 мин
шаг 800/3,890 | loss 0.4673 | 15.7 мин, осталось ~61 мин
шаг 900/3,890 | loss 0.4767 | 17.7 мин, осталось ~59 мин
шаг 1,000/3,890 | loss 0.4673 | 19.7 мин, осталось ~57 мин
шаг 1,100/3,890 | loss 0.4760 | 21.7 мин, осталось ~55 мин
шаг 1,200/3,890 | loss 0.4681 | 23.6 мин, осталось ~53 мин
шаг 1,300/3,890 | loss 0.4656 | 25.6 мин, осталось ~51 мин
шаг 1,400/3,890 | loss 0.4585 | 27.6 мин, осталось ~49 мин
шаг 1,500/3,890 | loss 0.4505 | 29.6 мин, осталось ~47 мин
шаг 1,600/3,890 | loss 0.4587 | 31.5 мин, осталось ~45 мин
шаг 1,700/3,890 | loss 0.4393 | 33.5 мин, осталось ~43 мин
шаг 1,800/3,890 | lo

## 7. Векторы раунда 2 и сравнение с раундом 1

In [9]:
item_emb = encode(corpus.p_text, P_MAX_LEN, desc='корпус: полный текст')
item_title_emb = encode(corpus.t_text, T_MAX_LEN, desc='корпус: заголовки')
need_q = queries[queries.qid.isin(cb_train_qids | val_qids) | (queries.split == 'bench')].reset_index(drop=True)
query_emb = encode(need_q.q_text, Q_MAX_LEN, desc='запросы (CatBoost-train, val, бенчмарк)')
q_pos = pd.Series(np.arange(len(need_q)), index=need_q.qid)
EVAL['раунд 2'] = evaluate(item_emb, query_emb[q_pos[vq.qid].values], 'раунд 2')
res = pd.DataFrame(EVAL).T
display(res.style.format('{:.4f}'))
d50 = res.loc['раунд 2', 'dense@50'] - res.loc['раунд 1', 'dense@50']
dc = res.loc['раунд 2', 'потолок BM25+dense'] - res.loc['раунд 1', 'потолок BM25+dense']
print(f'изменение: dense@50 {d50:+.4f}, потолок BM25+dense {dc:+.4f} -> '
      f'{"есть смысл пересобирать признаки v2 и CatBoost" if d50 >= 0.01 else "прирост мал — дальше по цепочке идти не стоит"}')

  корпус: полный текст: 189,212 текстов за 458 с
  корпус: заголовки: 189,212 текстов за 71 с
  запросы (CatBoost-train, val, бенчмарк): 20,452 текстов за 8 с
[раунд 2] dense@50: 0.8409 | dense@100: 0.8902 | потолок BM25: 0.9322 | потолок BM25+dense: 0.9693


,dense@50,dense@100,потолок BM25,потолок BM25+dense
раунд 1,0.8359,0.8846,0.9322,0.9673
раунд 2,0.8409,0.8902,0.9322,0.9693


изменение: dense@50 +0.0050, потолок BM25+dense +0.0020 -> прирост мал — дальше по цепочке идти не стоит


## 8. Сохранение (те же имена файлов, что у раунда 1)

In [10]:
np.save(f'{OUT_DIR}/item_emb.npy', item_emb)
np.save(f'{OUT_DIR}/item_title_emb.npy', item_title_emb)
np.save(f'{OUT_DIR}/query_emb.npy', query_emb)
pd.DataFrame({'item_id': corpus.item_id}).to_parquet(f'{OUT_DIR}/item_ids.parquet', index=False)
pd.DataFrame({'qid': need_q.qid}).to_parquet(f'{OUT_DIR}/query_ids.parquet', index=False)
encoder.bert.save_pretrained(f'{OUT_DIR}/model')
tokenizer.save_pretrained(f'{OUT_DIR}/model')
json.dump({'round': 2, 'init': MODEL1_DIR, 'hard_negatives': [NEG_FROM, NEG_TO], 'train_pairs': len(tr), 'steps': step,
           'batch_size': BATCH_SIZE, 'lr': LR, 'eval': EVAL},
          open(f'{OUT_DIR}/meta.json', 'w'), ensure_ascii=False, indent=2)
print(sorted(os.listdir(OUT_DIR)))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['item_emb.npy', 'item_ids.parquet', 'item_title_emb.npy', 'meta.json', 'model', 'query_emb.npy', 'query_ids.parquet']
